# Train your own Pleiades SVM model

This notebook loads QuPath-exported ground-truth ROIs (image + label mask + roi_mapping.csv) (from `QuPath_script_1.txt`), computes features via `roi_metrics()` (from `BlueNuclei_utils.py`), builds a single training DataFrame `df_all`, and trains & save an SVM model.

Outputs:
An SVM model and supporting scaler files (all in .pkg format).

Notes:

>This expects the QuPath export layout: `.../<folder>/ground_truth/images/*.tif`, `masks/*.tif`, `roi_mapping.csv`.

>You need to have the BlueNuclei_utils.py file.


In [7]:
import os
import numpy as np
import pandas as pd
import tifffile
from tqdm import tqdm
from scipy.ndimage import gaussian_filter, laplace

# Import your feature extractor (from your BlueNuclei codebase)
# from bluenuclei_utils import roi_metrics

# If roi_metrics is already in your environment via another import, you can delete the line above.


In [ ]:
# ---- Configure paths ----

# macOS example:
root_dir = r"/Users/your_file_path"

# Dataset folders under root_dir
# Each folder contains a training image. Adjust the names if needed.
data_folders = ['a1', 'b1', 'c1', 'd1', 'e1', 'f1', 't1', 't5', 't10', 't15', 't20']


In [ ]:
def pick_single_tif(folder_path: str) -> tuple[str, str]:
    """Pick one image tif and one mask tif from a QuPath export folder.

    Assumes exactly one .tif in each of:
      - ground_truth/images
      - ground_truth/masks

    If you have multiple images per folder, replace this with a basename-matching strategy.
    """
    image_path = os.path.join(folder_path, 'images')
    mask_path = os.path.join(folder_path, 'masks')

    img_files = sorted([f for f in os.listdir(image_path) if f.lower().endswith('.tif')])
    mask_files = sorted([f for f in os.listdir(mask_path) if f.lower().endswith('.tif')])

    if len(img_files) == 0:
        raise FileNotFoundError(f"No .tif found in {image_path}")
    if len(mask_files) == 0:
        raise FileNotFoundError(f"No .tif found in {mask_path}")

    # Default: first tif (mirrors your current notebook logic)
    return os.path.join(image_path, img_files[0]), os.path.join(mask_path, mask_files[0])


In [ ]:
# ---- Build training rows from QuPath-exported ROIs ----
feature_rows = []

for folder in tqdm(data_folders, desc="Processing folders"):
    folder_path = os.path.join(root_dir, folder, 'ground_truth')
    csv_path = os.path.join(folder_path, 'roi_mapping.csv')

    img_fp, mask_fp = pick_single_tif(folder_path)

    # Load image + label mask
    img = tifffile.imread(img_fp)
    mask = tifffile.imread(mask_fp)

    # Edge image used by your roi_metrics()
    blurred = gaussian_filter(img, sigma=4.5)
    laplacian = np.abs(laplace(blurred))

    # ROI mapping from QuPath export
    roi_df = pd.read_csv(csv_path)

    # Normalize intensity by whole-image median (your current approach)
    median_dapi = float(np.median(img))

    for _, row in roi_df.iterrows():
        obj_id = row['Object ID']
        roi_value = row['Label']

        # Your convention: D* => dead, otherwise live (because QuPath script renames to D1/D2 and L1/L2)
        roi_class = 'dead' if str(obj_id).upper().startswith('D') else 'live'

        ys, xs = np.where(mask == roi_value)
        if len(xs) == 0:
            continue

        values = img[ys, xs]
        roi_pixels_df = pd.DataFrame({'X': xs, 'Y': ys, 'Value': values})

        metrics_df = roi_metrics(roi_pixels_df, laplacian, edge_scale=0.7)
        if metrics_df is None or metrics_df.empty:
            continue

        feature_row = metrics_df.iloc[0].copy()
        feature_row['intensity'] = feature_row['intensity'] / median_dapi
        feature_row['label'] = roi_class
        feature_row['folder'] = folder
        feature_rows.append(feature_row)

# Final combined training table
df_all = pd.DataFrame(feature_rows)
df_all


In [ ]:
# ---- Train + save an SVM model ----

import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

# ---- (Optional) guard ----
required_cols = {'folder','label','spottiness','distribution','intensity','area','edge_gradient'}
missing = sorted(required_cols - set(df_all.columns))
if missing:
    raise ValueError(f"df_all is missing required columns: {missing}")

confirm = input("Type 'train' to confirm training an SVM model: ").strip().lower()
if confirm != "train":
    print("Canceled.")
else:
    # === Step 1: Domain map (A vs T) ===
    domain_map = {
        'a1': 'A', 'b1': 'A', 'c1': 'A', 'd1': 'A', 'e1': 'A', 'f1': 'A',
        't1': 'T', 't5': 'T', 't10': 'T', 't15': 'T', 't20': 'T'
    }
    df_all = df_all.copy()
    df_all['domain'] = df_all['folder'].map(domain_map)

    if df_all['domain'].isna().any():
        bad_folders = df_all.loc[df_all['domain'].isna(), 'folder'].unique().tolist()
        raise ValueError(f"Some folders were not mapped to a domain (A/T): {bad_folders}")

    features = ['spottiness', 'distribution', 'intensity', 'area', 'edge_gradient']

    df_A = df_all[df_all['domain'] == 'A'].copy()
    df_T = df_all[df_all['domain'] == 'T'].copy()

    # === Step 2: Compute affine alignment stats (T → A) ===
    A_means = df_A[features].mean()
    A_stds  = df_A[features].std(ddof=1).replace(0, np.nan)

    T_means = df_T[features].mean()
    T_stds  = df_T[features].std(ddof=1).replace(0, np.nan)

    # Apply alignment: T -> A
    df_T.loc[:, features] = ((df_T[features] - T_means) / T_stds) * A_stds + A_means
    df_all_aligned = pd.concat([df_A, df_T], ignore_index=True)

    # If any feature ended up NaN due to zero std, fail fast
    if df_all_aligned[features].isna().any().any():
        nan_cols = df_all_aligned[features].columns[df_all_aligned[features].isna().any()].tolist()
        raise ValueError(
            f"NaNs detected after alignment in columns {nan_cols}. "
            f"Likely a zero std in A/T stats; inspect A_stds/T_stds."
        )

    # === Step 3: Prepare X/Y ===
    X = df_all_aligned[features].copy()
    Y = df_all_aligned['label'].astype(str)

    # Encode label: le.classes_ will define which side of decision_function is class 1
    le = LabelEncoder()
    Y_enc = le.fit_transform(Y)

    # === Step 4: Scale features (match your production pipeline) ===
    std_scaler = StandardScaler()
    minmax_scaler = MinMaxScaler()

    X_scaled = X.copy()
    X_scaled[['spottiness', 'distribution', 'intensity', 'edge_gradient']] = std_scaler.fit_transform(
        X_scaled[['spottiness', 'distribution', 'intensity', 'edge_gradient']]
    )
    X_scaled[['area']] = minmax_scaler.fit_transform(X_scaled[['area']])

    # === Step 5: Train/test split + train LinearSVC ===
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, Y_enc, test_size=0.2, random_state=42, stratify=Y_enc
    )

    model = LinearSVC(C=5.0, max_iter=20000, dual=False, class_weight='balanced')
    model.fit(X_train, y_train)

    # === Step 6: Apply your fixed threshold and report ===
    # NOTE: decision_function is for "class 1" of LabelEncoder order.
    decision_scores = model.decision_function(X_test)

    best_threshold = -1.5  # your chosen threshold
    y_pred_adjusted = (decision_scores > best_threshold).astype(int)

    print("\nLabelEncoder classes_ order:", le.classes_)
    print("\nclassification report (decision_score > threshold => class index 1):")
    print(classification_report(y_test, y_pred_adjusted, target_names=le.classes_))

    # === Step 7: Save artifacts ===
    # (save to current working directory; change paths if you want)
    joblib.dump(model, 'svm_model.pkl')
    joblib.dump(std_scaler, 'std_scaler.pkl')
    joblib.dump(minmax_scaler, 'minmax_scaler.pkl')
    joblib.dump(le, 'label_encoder.pkl')
    joblib.dump(best_threshold, 'svm_threshold.pkl')

    # Optional but recommended: save domain-alignment stats used during training
    # so inference can align T-domain samples the same way later.
    joblib.dump(A_means, 'A_means.pkl')
    joblib.dump(A_stds, 'A_stds.pkl')
    joblib.dump(T_means, 'T_means.pkl')
    joblib.dump(T_stds, 'T_stds.pkl')

    print("\nSaved: svm_model.pkl, std_scaler.pkl, minmax_scaler.pkl, label_encoder.pkl, svm_threshold.pkl")
    print("Also saved (recommended): A_means.pkl, A_stds.pkl, T_means.pkl, T_stds.pkl")
